# SFT rồi DPO cho bài trích xuất phản hồi sinh viên

Module 11 gọi `qwen2.5:3b` bằng prompt để phân loại câu phản hồi UIT-VSFC. Ở đây bạn làm cách ngược lại: lấy model
nhỏ hơn 6 lần (`Qwen2.5-0.5B-Instruct`), **fine-tune** nó bằng LoRA để trả đúng một object JSON
`{"sentiment": ..., "topic": ...}`, rồi thử DPO trên chính các lỗi của model sau SFT. Mọi thước đo đều kiểm bằng code,
không cần LLM chấm: tỉ lệ output đúng định dạng và macro-F1 trên tập ghim 150 câu của module 11 và cả tập test.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) tải model, train, đo; test bỏ qua chúng.
- Chạy thật bằng kernel `ai-gpu` (GPU 4GB là đủ). Kernel CPU chỉ chạy bản thu nhỏ để kiểm code chạy hết, con số không có ý nghĩa.
- Chấm: `uv run pytest tracks/ai-engineer/modules/14-fine-tuning-llm/exercises/intermediate/03-sft-dpo-feedback -q`

In [ ]:
# explore: tìm gốc repo, tải UIT-VSFC nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


VSFC = dataset_dir("uit-vsfc")
CACHE = VSFC / "cache"
CACHE.mkdir(exist_ok=True)
print(VSFC)

In [ ]:
import json
import re

import torch

## 1. Định dạng hội thoại, đọc output, macro-F1

Hằng `SYSTEM`, `SENTIMENTS`, `TOPICS` đã cho. Viết:

- `to_chat(sentence, sentiment=None, topic=None) -> list[dict]`: message `system` (`SYSTEM`), message `user` (chính câu
  phản hồi). Có nhãn thì thêm message `assistant` có nội dung là **đúng chuỗi**
  `json.dumps({"sentiment": ..., "topic": ...}, ensure_ascii=False)` (nhãn là tên trong `SENTIMENTS`, `TOPICS`).
- `parse_output(text, strict=True) -> tuple[str, str] | None`: `strict=True` thì cả chuỗi (bỏ khoảng trắng hai đầu) phải
  là một object JSON có đúng hai khoá `sentiment`, `topic`, giá trị thuộc `SENTIMENTS`, `TOPICS`. `strict=False` thì
  lấy object `{...}` đầu tiên (không lồng) xuất hiện trong chuỗi rồi kiểm như trên. Sai ở bất kỳ đâu trả `None`.
- `macro_f1(y_true, y_pred, labels) -> float`: trung bình F1 của các lớp trong `labels`; dự đoán `None` (output hỏng)
  tính là sai. Lớp không xuất hiện ở cả hai phía có F1 bằng 0.

In [ ]:
SYSTEM = ("Bạn phân loại câu phản hồi của sinh viên về môn học. Trả về đúng một object JSON với hai khoá: "
          '"sentiment" là một trong "negative", "neutral", "positive"; "topic" là một trong "lecturer", '
          '"training_program", "facility", "others". Không viết gì khác.')
SENTIMENTS = ["negative", "neutral", "positive"]
TOPICS = ["lecturer", "training_program", "facility", "others"]


def to_chat(sentence: str, sentiment: str | None = None, topic: str | None = None) -> list[dict]:
    # TODO
    raise NotImplementedError


def parse_output(text: str, strict: bool = True) -> tuple[str, str] | None:
    # TODO
    raise NotImplementedError


def macro_f1(y_true: list[str], y_pred: list[str | None], labels: list[str]) -> float:
    # TODO
    raise NotImplementedError

## 2. Sinh theo batch và cặp ưu tiên

- `generate_batch(model, tokenizer, chats, max_new_tokens=32, batch_size=32) -> list[str]`: greedy, mỗi chat dựng
  prompt bằng `apply_chat_template(..., add_generation_prompt=True)`; trả phần **mới sinh** đã decode (bỏ special
  token), đúng thứ tự `chats`. Batch có độ dài khác nhau phải cho **cùng kết quả** như sinh từng câu một.
- `preference_pairs(sentences, gold, outputs) -> list[dict]`: với mỗi câu mà `parse_output(output)` (strict) khác nhãn
  vàng `gold[i] = (sentiment, topic)`, tạo một cặp theo định dạng hội thoại của TRL:
  `{"prompt": to_chat(câu), "chosen": [assistant: JSON vàng], "rejected": [assistant: output của model]}`. Câu model
  đã đúng thì bỏ qua.

In [ ]:
@torch.no_grad()
def generate_batch(model, tokenizer, chats: list[list[dict]], max_new_tokens: int = 32,
                   batch_size: int = 32) -> list[str]:
    """Greedy theo batch, trả phần mới sinh đã decode, đúng thứ tự chats."""
    # TODO
    raise NotImplementedError


def preference_pairs(sentences: list[str], gold: list[tuple[str, str]], outputs: list[str]) -> list[dict]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu: tập ghim 150 câu của module 11, tập test đầy đủ, chia train cho SFT và cho DPO
import time

import numpy as np
import pandas as pd

train = pd.read_parquet(VSFC / "default/train/0000.parquet")
test = pd.read_parquet(VSFC / "default/test/0000.parquet")
parts = [test[test.sentiment == c].sample(50, random_state=0) for c in range(3)]
pinned = pd.concat(parts).sample(frac=1.0, random_state=0)
order = np.random.default_rng(0).permutation(len(train))
GPU = torch.cuda.is_available()
SFT_N, POOL_N = (8000, 3426) if GPU else (48, 24)
sft_rows = train.iloc[order[:SFT_N]]
pool_rows = train.iloc[order[SFT_N: SFT_N + POOL_N]]
test_rows = test if GPU else pinned.iloc[:24]
pinned_rows = pinned if GPU else pinned.iloc[:24]


def gold_of(df):
    return [(SENTIMENTS[s], TOPICS[t]) for s, t in zip(df.sentiment, df.topic)]


print(f"{'GPU' if GPU else 'CPU (bản thu nhỏ)'}: SFT {len(sft_rows)} câu, pool DPO {len(pool_rows)}, "
      f"ghim {len(pinned_rows)}, test {len(test_rows)}")
print(to_chat(sft_rows.sentence.iloc[0], *gold_of(sft_rows)[0])[-1])

In [ ]:
# explore: nạp Qwen2.5-0.5B-Instruct (lần đầu tải khoảng 1 GB), đo model gốc chỉ bằng prompt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if GPU else "cpu"
DTYPE = torch.bfloat16 if GPU else torch.float32
torch.manual_seed(0)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE).eval()


def evaluate(model, name, batch_size=64):
    t0 = time.perf_counter()
    res = {}
    for split, df in (("pinned", pinned_rows), ("test", test_rows)):
        outs = generate_batch(model, tokenizer, [to_chat(s) for s in df.sentence], batch_size=batch_size)
        gold = gold_of(df)
        strict = [parse_output(o) for o in outs]
        lenient = [parse_output(o, strict=False) for o in outs]
        res[split] = {
            "n": len(df), "strict_ok": sum(p is not None for p in strict) / len(df),
            "lenient_ok": sum(p is not None for p in lenient) / len(df),
            "f1_sentiment": macro_f1([g[0] for g in gold], [p[0] if p else None for p in strict], SENTIMENTS),
            "f1_topic": macro_f1([g[1] for g in gold], [p[1] if p else None for p in strict], TOPICS),
            "f1_sentiment_lenient": macro_f1([g[0] for g in gold], [p[0] if p else None for p in lenient], SENTIMENTS),
            "example": outs[0],
        }
    res["eval_seconds"] = round(time.perf_counter() - t0, 1)
    print(name, json.dumps(res, ensure_ascii=False))
    return res


results = {"device": torch.cuda.get_device_name() if GPU else "cpu", "base": evaluate(base, "base")}

In [ ]:
# explore: SFT bằng LoRA với TRL: chỉ tính loss trên phần completion (JSON), đo thời gian và VRAM đỉnh
import tempfile

from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def lora_config():
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")


def sft_dataset(df):
    return Dataset.from_list([{"prompt": to_chat(s), "completion": to_chat(s, *g)[-1:]}
                              for s, g in zip(df.sentence, gold_of(df))])


def train_args(cls, **kw):
    common = dict(output_dir=tempfile.mkdtemp(prefix="m14-trl-"), per_device_train_batch_size=16, num_train_epochs=1,
                  logging_steps=50,
                  save_strategy="no", report_to=[], bf16=GPU, use_cpu=not GPU, seed=0, data_seed=0,
                  dataloader_num_workers=0, max_length=256)
    common.update(kw)
    return cls(**common)


def run_sft(model, df, name, lr=2e-4):
    if GPU:
        torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(model=model, args=train_args(SFTConfig, learning_rate=lr,
                                                      lr_scheduler_type="cosine", warmup_steps=0.03),
                         train_dataset=sft_dataset(df), processing_class=tokenizer, peft_config=lora_config())
    t0 = time.perf_counter()
    out = trainer.train()
    stats = {"steps": out.global_step, "train_loss": round(out.training_loss, 4),
             "seconds": round(time.perf_counter() - t0, 1),
             "peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2) if GPU else None,
             "trainable": sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)}
    print(name, stats)
    return trainer.model.merge_and_unload().eval(), stats


sft_model, results["sft_train"] = run_sft(AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE),
                                          sft_rows, "sft")
results["sft"] = evaluate(sft_model, "sft")

In [ ]:
# explore: cặp ưu tiên từ lỗi của chính model SFT trên pool, rồi DPO (reference = model SFT)
from trl import DPOConfig, DPOTrainer

pool_out = generate_batch(sft_model, tokenizer, [to_chat(s) for s in pool_rows.sentence], batch_size=64)
pairs = preference_pairs(list(pool_rows.sentence), gold_of(pool_rows), pool_out)
results["pairs"] = {"pool": len(pool_rows), "pairs": len(pairs),
                    "invalid_rejected": sum(parse_output(p["rejected"][0]["content"]) is None for p in pairs)}
print(results["pairs"])
print(pairs[0])


def run_dpo(model, pairs, beta, lr=5e-5, epochs=2):
    if GPU:
        torch.cuda.reset_peak_memory_stats()
    args = train_args(DPOConfig, beta=beta, learning_rate=lr, num_train_epochs=epochs,
                      per_device_train_batch_size=8, lr_scheduler_type="cosine", warmup_steps=0.1, logging_steps=10)
    trainer = DPOTrainer(model=model, args=args, train_dataset=Dataset.from_list(pairs), processing_class=tokenizer,
                         peft_config=lora_config())
    t0 = time.perf_counter()
    out = trainer.train()
    logs = [h for h in trainer.state.log_history if "rewards/accuracies" in h]
    stats = {"beta": beta, "lr": lr, "steps": out.global_step, "train_loss": round(out.training_loss, 4),
             "seconds": round(time.perf_counter() - t0, 1),
             "peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2) if GPU else None,
             "first_log": logs[0] if logs else None, "last_log": logs[-1] if logs else None}
    print(stats)
    return trainer.model.merge_and_unload().eval(), stats


dpo_runs = {}
for lr in ((5e-5, 5e-6) if GPU else (5e-6,)):
    sft_copy = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE)
    sft_copy.load_state_dict(sft_model.state_dict())
    dpo_model, train_stats = run_dpo(sft_copy, pairs, beta=0.1, lr=lr)
    dpo_runs[f"{lr:g}"] = {"train": train_stats, "eval": evaluate(dpo_model, f"dpo beta=0.1 lr={lr:g}")}
    del dpo_model, sft_copy
results["dpo"] = dpo_runs

In [ ]:
# explore: đối chứng cùng ngân sách dữ liệu: SFT thêm trên đúng pool đó (nhãn vàng của mọi câu, không chỉ câu sai)
sft_copy = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEVICE)
sft_copy.load_state_dict(sft_model.state_dict())
more_model, results["sft_more_train"] = run_sft(sft_copy, pool_rows, "sft_more")
results["sft_more"] = evaluate(more_model, "sft_more")
del more_model, sft_copy
path = CACHE / ("m14-sft-dpo.json" if GPU else "m14-sft-dpo-cpu.json")
path.write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")
print("đã ghi", path)

In [ ]:
# explore: bảng tổng hợp (khi chạy CPU, in thêm kết quả GPU đã cache nếu có)
def table(res):
    rows = {"Gốc, chỉ prompt": res["base"], "SFT LoRA": res["sft"], "SFT + SFT thêm trên pool": res["sft_more"]}
    rows.update({f"SFT + DPO lr={k}": r["eval"] for k, r in res["dpo"].items()})
    return pd.DataFrame({name: {"ghim: JSON đúng (strict)": r["pinned"]["strict_ok"],
                                "ghim: F1 cảm xúc": r["pinned"]["f1_sentiment"],
                                "test: JSON đúng (strict)": r["test"]["strict_ok"],
                                "test: F1 cảm xúc": r["test"]["f1_sentiment"],
                                "test: F1 chủ đề": r["test"]["f1_topic"]} for name, r in rows.items()}).T.round(3)


display(table(results))
gpu_cache = CACHE / "m14-sft-dpo.json"
if not GPU and gpu_cache.exists():
    print("Kết quả GPU đã cache:")
    display(table(json.loads(gpu_cache.read_text(encoding="utf-8"))))

## Đọc kết quả

So với module 11 (`qwen2.5:3b` chỉ bằng prompt trên cùng 150 câu ghim, cùng thước đo macro-F1 cảm xúc), tự trả lời:

- Model gốc 0.5B có trả đúng định dạng không? Nó hỏng kiểu gì (xem `example`), và `strict` khác `lenient` bao nhiêu?
- Sau SFT, F1 cảm xúc trên 150 câu ghim so với model 3B dùng prompt thế nào? Trên tập test đầy đủ (lệch lớp) thì sao?
- DPO trên các lỗi của model SFT đổi được gì so với SFT thêm trên **cùng** pool? Hai learning rate cho kết quả thế nào?
  Nhìn `rewards/margins`, `logps/chosen` ở log đầu và cuối của mỗi lượt: reward ngầm tăng có đồng nghĩa với model tốt hơn không?
- Đọc vài cặp ưu tiên: nhãn vàng của UIT-VSFC có luôn đúng không? DPO đẩy model về phía nhãn vàng ở **mọi** cặp.
- Lớp `neutral` hiếm trong train (khoảng 4%) nhưng chiếm một phần ba tập ghim: nó kéo F1 cảm xúc thế nào?

Sau đó làm phần trắc nghiệm của bài trong app.